# Modelo 8 — Isolation Forest

Detector de anomalias nas portas de plataforma (PSDs).

Diferente do Modelo 5, este notebook **não prevê ZM01**. O Isolation Forest é não supervisionado: ele marca o dia da porta cujo padrão de alarmes e manutenção foge do histórico. O rótulo `y` (falha em 7 dias) entra só no final, para ver se anomalia e quebra coincidem.

Reusa o painel porta × dia do Modelo 5. Se `painel.parquet` não existir, rode antes `models/modelo-5/construir_painel.py`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score, classification_report
from sklearn.preprocessing import RobustScaler

raiz = Path.cwd()
if not (raiz / 'models' / 'modelo-5' / 'painel.parquet').exists():
    for pasta in Path.cwd().parents:
        candidato = pasta / 'models' / 'modelo-5' / 'painel.parquet'
        if candidato.exists():
            raiz = pasta
            break

painel_path = raiz / 'models' / 'modelo-5' / 'painel.parquet'
painel_path

## 1. Painel

Uma linha por porta por dia. As features de alarme e SAP já vêm defasadas em 1 dia (sem vazamento do próprio dia nem do futuro).

In [ ]:
painel = pd.read_parquet(painel_path)
painel['Data'] = pd.to_datetime(painel['Data'])

exclui = {
    'id_porta', 'Data', 'est', 'via', 'id_plat', 'y',
    'n_falhas_7d_futuro', 'cobertura_30d', 'mes',
}
feats = [c for c in painel.columns if c not in exclui and pd.api.types.is_numeric_dtype(painel[c])]

print(painel.shape)
print('portas / estações:', painel['id_porta'].nunique(), '/', painel['est'].nunique())
print('taxa ZM01 em 7d:', f"{painel['y'].mean() * 100:.2f}%")
print('n features:', len(feats))
painel.head()

## 2. Split temporal

Mesmo recorte do Modelo 5: treino até 23/04/2025, teste a partir de 01/05/2025.

In [ ]:
FIM_TREINO = pd.Timestamp('2025-04-23')
INI_TESTE = pd.Timestamp('2025-05-01')

treino = painel[painel['Data'] <= FIM_TREINO].copy()
teste = painel[painel['Data'] >= INI_TESTE].copy()


def montar_X(df):
    X = df[feats].replace([np.inf, -np.inf], np.nan)
    X = X.fillna({c: 0 for c in feats if str(c).startswith(('n_', 'zm0', 'dur_'))})
    for c in ('dias_desde_zm01', 'dias_desde_zm02', 'dias_desde_zm03'):
        if c in X.columns:
            X[c] = X[c].fillna(9999)
    return X.fillna(X.median(numeric_only=True))


X_tr, y_tr = montar_X(treino), treino['y'].astype(int).to_numpy()
X_te, y_te = montar_X(teste), teste['y'].astype(int).to_numpy()
print('treino:', len(X_tr), '| teste:', len(X_te), '| taxa ZM01 treino:', f'{y_tr.mean() * 100:.2f}%')

## 3. Isolation Forest

`contamination` usa a taxa de ZM01 do treino só para definir quantos pontos o `predict` marca como `-1`. O algoritmo em si não vê o rótulo.

Score de anomalia = `-score_samples` (quanto maior, mais atípico).

In [ ]:
scaler = RobustScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

contaminacao = float(np.clip(y_tr.mean(), 0.01, 0.49))
iso = IsolationForest(
    n_estimators=200,
    max_samples=min(2048, len(X_tr)),
    contamination=contaminacao,
    random_state=42,
    n_jobs=-1,
)
iso.fit(X_tr_s)

score_te = -iso.score_samples(X_te_s)
pred_te = (iso.predict(X_te_s) == -1).astype(int)

print('contamination:', round(contaminacao, 4))
print('PR-AUC (anomalia vs ZM01 em 7d):', round(average_precision_score(y_te, score_te), 4))
print(classification_report(y_te, pred_te, digits=3, zero_division=0))